# Introduction

This notebook re-implements the [source code](https://github.com/oballinger/PWTT) of [Open access battle damage detection via Pixel-Wise T-Test on Sentinel-1 imagery](https://doi.org/10.1016/j.rse.2025.115025) with specific changes like running in local environmnet, filtering the damage outcome with additional layers like ESA Landcover 2021 or GHSL buildup data.

## Manage the path

- First attach the path where modified code and notebook is saved. Bear in mind that this enables notebook to load modified code to the memory
- Import the modified code 
- Import necessary packages for visualization and additional wraper functions implemented in this notebook

## Citations and disclaimer 
- Ensure to use the right citation, Ballinger, O. PWTT. GitHub repository. https://github.com/oballinger/PWTT to mention the tresholds and other parameters that might  explain implicit limitation or strenght of the product.
- Even if the data are open source we recommend to keep the product for internal use. 

GeoHum - MSF Gella  Gella Getachew Workineh Yann Rebois - June 2026

In [1]:
import sys
sys.path.append("E:/RESOURCES/PYTHON/envs/p314_geohumpwtt/modifiedscripts")  ## path to folder where you saved modified code and jupyter lab

In [2]:
import ee
from pwtt_modified import detect_damage   ## This is mainly importing from local file

In [3]:
## additional libraries for further customized wraper functions 
import rasterio as rio
import geopandas as gpd
from rasterio.features import geometry_window
from shapely.geometry import box, mapping
from glob import glob

## Authenticate Google Earth Engine
This is important to use Google earth Engine compute and builtin functionality. Please note that though the implementation is coded using python wraper, all the data acess and conputations are happening using Google Earth Engine builtin functionality and compute in the backend 
- here change the value for **project_name**  to you google earth engine project name

In [4]:
project_name='even-trainer-274511'   # change to you gee project  actual one  -->yann
ee.Authenticate(auth_mode='localhost')
ee.Initialize(project=project_name)

## Damage detection and visualization
For the function "detect_damage" though there are many default arguments the following are very important to note:

- ```aoi```: this is the area of interest file definition. This should be always a Polygon geometry. This could be directly created by using ee.Rectangle, ee.Polygon, ee.Point followed wioth specific buffer or from predefined file (file bounding box (could be both raster and vector), or direct geometry file. This is provided in the cells below). The AOI should note be very high, especially if we are exporting to google drive, it takes substantial amount of time.
- ```war_start```: this is the reference point for the implemented statistical function to reference pre images (from this time backwards is supposed to be normal images without any damage signal)
-  ```inference_start```: this is to consider from which onwards the images are supposed to be post images with damage signal
- ```pre_interval```: the number of months to consider before the war_start
- ```post_interval```: the number of months to consider after the inference start. Please note that this should be a minimum of 1 (this gives a model at least to have some images after the time frame where we want to do an inference). Please note that specifiying a minimum of 1 or greater does not guarantee availability of the image which is always governed by the time gap between inference start date. Assume if the inference starts one or two days before today, what ever post interval we put, the probability of getting image within 2-3 days interval is quite small.
- ```filter_builtup```: this is to further refine the obtained results using a built-up layer. The arguments could be one of 
    1. ```None```: do not perform any kind of spatial filtering. It should be noted that in the backend the function uses **urban** class from Dynamic World Dataset as a default spatial filtering layer. More information on [Dynamic World V1](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1)
    2. ```esa_cover```: perform spatial filtering with **builtup** class of the European Space Agency (ESA) worldcover dataset. More documentation on [ESA WorldCover 10m v200](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200)
    3. ```google_cover```: performs spatial filtering with **builtup_presence** layer of google building temporal dataset where building presence probability greater than or equal is set to 0.1. For more documentation please consult here. Please note that though this product is very high resolution, in some geographies its void. More documentation on [Open Buildings Temporal V1](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_Research_open-buildings-temporal_v1#description)
    4. ```ghs_cover```: perform spatial filtering with **builtup** class of Global Human settlement layer 2023 with per pixel builtup area greater than 0. See the more dicumentation [GHSL: Global built-up surface 1975-2030 (P2023A)](https://developers.google.com/earth-engine/datasets/catalog/JRC_GHSL_P2023A_GHS_BUILT_S#description)
    5. ```ghs_pop```: perform spatial filtering with **population_count** class of Global Human settlement layer 2023 with per pixel absolute number of resident population  greater than 1. Note that when a threshold of grater than 0 is indicated the outcome is the same as ```ghs_cover``` that the layer should be used with caution.  See the more dicumentation [GHSL: Global population surfaces 1975-2030 (P2023A))](https://developers.google.com/earth-engine/datasets/catalog/JRC_GHSL_P2023A_GHS_POP#description)
- ```threshold```: the statistical threshold to enforce above which is a damaged and the revers is background. Here the outcome from this is binary damage layer. The default value provided in the [original implementation](https://doi.org/10.1016/j.rse.2025.115025) was 3.3. If the values are greater than this it will give less false positives while the reverse provides more false positives. So, experimenting with different values before exporting is highly recomended- with our tests in Lebanon it is recommended to go to the higher tresholds of 5 minimum.
- ```viz```: whether to visualize the output within a notebook. if specified **True** it plots the T-statistic plot overlaid with basemap. Please note that if it plots, the results are not exported- recommended to put it as false
- ```export```: whether export the results to google drive
- ```footprints```: whether we have building foot prints to do zonal statistics and directly classify the damage class. The argument value could be of the following. It should be noted that if this option is specified other than ```None```, we have to watch the size of the area given Google flags maximum number of requests error.
    1. ```None```: do not do any kind of zonal statistics
    2. ```filepath```: this should be geometry file as a json file containing building geometries 
    3. ```google```: directly takes building footpints with higher confidens (>=0.5) from google buildings v3 product. See the documentation here . Please note that in some geographies this product is void. More information on the product using the following documentation on [Open Buildings V3 Polygons](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_Research_open-buildings_v3_polygons)
- ```export_name```: the name to export the file 


**To understand the effect of spatial scale, are size of zonal aggregation you can have a look or play around with [Google Earth Engine Code Snippet](https://code.earthengine.google.com/4b5cf4f13c03510bf3ed9eb260eaf424)** 

### Rectangle geometry

In [ ]:
## Here you can chnage any argument you want to chnage based on provided explanation
# bbox for Tyr/sour
sour = ee.Geometry.Rectangle([35.09,33.05,35.56,33.33]) # the area of interest as a bounding box
detect_damage(aoi=sour,
                   war_start='2026-03-02',#YYYY-MM-DD  the start of the war
                   inference_start='2026-06-10', #YYYY-MM-DD the beginning of the inference window (meaning from when we should look at 
                   pre_interval=12,# the number of months before the war to use as a reference period 
                   post_interval=1, # the number of months after the war to use as a reference period 
                   viz=False,
                   export=True,
                   threshold=8,
                   filter_builtup=True,
                   export_name="LBN_Sour_DamagesProbability_PWTT_tresh8_strt0032026_end10062026")

### Point geometry with buffer

In [ ]:
## Here you can chnage any argument you want to chnage based on provided explanation
tehran = ee.Geometry.Point([51.37, 35.70]).buffer(20000)
detect_damage(aoi=tehran,
                   war_start='2026-03-01',
                   inference_start='2026-03-01',
                   pre_interval=12,
                   post_interval=2,
                   viz=False,
                   export=True,
                   threshold=4,
                   footprints=None,
                   filter_builtup=None,
                   export_name="export_name")

In [ ]:
westbank = ee.Geometry.Point([35.50045532568531,33.29145242279608]).buffer(1000)
out = detect_damage(aoi=westbank,
              war_start='2026-03-03',
              inference_start='2026-03-14',
              pre_interval=12,
              post_interval=2,
              viz=False,
              export=True,
              threshold=4,
              footprints=None,
              filter_builtup=None,
              export_name="export_name")

Exporting the image to drive----
Export successfull----->
{'state': 'READY', 'description': 'with_yann', 'priority': 100, 'creation_timestamp_ms': 1776843878743, 'update_timestamp_ms': 1776843878743, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': '4OALG2ZURJZIPK6M6NE3DVWE', 'name': 'projects/driven-atrium-401315/operations/4OALG2ZURJZIPK6M6NE3DVWE'}


# Geometry from the image/file bounding box and vector geometry

### Extracting geometry from files

On previous cells the function  ```detect_damage``` takes geometries created from user defined coordinates. Instead of user defined coordinates, the following function  ```file2geometry```, creates geometry from two types of files:
- raster --> mainly the bouding box geometry of the raster
- vector --> either the bounding box or the the exact geometry of the file

In [7]:
## Do not change anything in this cell, its a function definition always run!

def file2geometry(file, raster, use_vector_geometry):
    if raster:
        print(f"The bounding box geometry will be computed from raster file : {file}")
        rst = rio.open(file)
        bound = rst.bounds
        rst.close()
        geom = ee.Geometry.Rectangle(list(bound))
    else:
        fls = gpd.read_file(file)
        if not use_vector_geometry:
            print(f"The bounding box geometry will be computed from vector file : {file}")
            bound = fls.bounds
            geom = ee.Geometry.Rectangle(list(bound))
        else:
            print(f"The actual AOI geometry will be computed from vector file : {file}")
            cords = fls.geometry.get_coordinates().to_numpy().tolist()
            if fls.geometry.type == "MultiPolygon":
                geom = ee.Geometry.MultiPolygon(cords)
            elif fls.geometry.type == "Polygon":
                geom = ee.Geometry.Polygon(cords)
            else:
                    raise ValueError("Provided geometry is not either Polygon or MultiPolygon, please check input geometry file")
            
    return geom

In addition to previously defined in the followining code block, to use the function ```file2geometry``` we have the folowing argument inputs to the function
- ```file```: full path to either raster or vector file, where we want to extract geometry. 
- ```raster```: boolean value indicating whether the input is raster or vecor. If the file is raster, set it to ```True``` else set it to ```False```
- ```use_vector_geometry```: if the input file is vector where ```raster=False```, a boolean value if set to ```use_vector_geometry=True```, it uses vector file **exact boundary** and if its set to ```use_vector_geometry=False```, it uses vector file **bounding box** as a geometry


In [15]:
### Parameters to change as per your interest
# geometry file parameters
file = "E:/!TEMP/RD/GeoHum/Damages/PWTT/LBN_Sour_DamagesProbability_PWTT_tresh5_strt0032026_end10062026.tif" ## geometry file I:/PROCESSED/_MANUAL/LBN_BintJbeil_PNEO_22Apr2026/LBN_BintJbeil_PNEO_22Apr2026_TRC.tif
raster=True        ## whether the file is raster 
use_vector_geometry = False   ## whether to useexact file geometry instead of bounding box. This is mainy for vector file

# main function parameters 
iso3="LBN"
footprints=None               ## if you have any building footprints, see previous explanation
filter_builtup = "esa_cover"  ## see previous argument definitions and choices #  esa_cover , google_cover, ghs_cover ,ghs_pop
export=True                 ## if you want to export the outout, you have to set this True
viz=False                     ## if you want to export the outout, you have to set this False default False
war_start='2026-03-02'        ## when war started YYYY-MM-DD 
inference_start='2026-06-15'  ## from where inference should start YYYY-MM-DD 
pre_interval=12               ## time window backwards from war start in months
post_interval=1               ## time window forwards from inference start in months
threshold = 5.5                ## threshold to enforce on T-statistic, the larger the value the less false positives but with risk of false negatives and the revers holds true 

#for the export 
builtup_map = {
    "esa_cover": "ESA2021",
    "google_cover": "GoogleTemporal",
    "ghs_cover": "GHS2023",
    "ghs_pop": "GHSPoP"
}
builtup_tag = builtup_map.get(filter_builtup, filter_builtup.upper())

war_start_fmt = war_start.replace("-", "")
inference_start_fmt = inference_start.replace("-", "")

aoi_name = (
    f"{iso3}"
    f"_Sour_"#optional
    f"{builtup_tag}_"
    f"PWTT_"
    f"tresh{threshold}_"
    f"strt{war_start_fmt}_"
    f"end{inference_start_fmt}"
)
aoi_name



#aoi_name = "LBN_Sour_DamagesProbability_PWTT_tresh6_strt0032026_end15062026"

# do not change anything below, as changes should be done above

aoi = file2geometry(file=file, raster=raster, use_vector_geometry=use_vector_geometry)  ## uses a function defined above and extract geometry

detect_damage(aoi=aoi,
            war_start=war_start,
            inference_start=inference_start,
            pre_interval=pre_interval,
            post_interval=post_interval,
            filter_builtup=filter_builtup,
            viz=viz,
            export=export,
            threshold=threshold,
            footprints=footprints, 
            export_name=f"{aoi_name}")

The bounding box geometry will be computed from raster file : E:/!TEMP/RD/GeoHum/Damages/PWTT/LBN_Sour_DamagesProbability_PWTT_tresh5_strt0032026_end10062026.tif
Exporting the image to drive----
Export successfull----->
{'state': 'READY', 'description': 'LBN_Sour_ESA_2021_PWTT_tresh5.5_strt20260302_end20260615', 'priority': 100, 'creation_timestamp_ms': 1782136151722, 'update_timestamp_ms': 1782136151722, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'DLDNXVBCWERBB2CY5WQEY3ZW', 'name': 'projects/even-trainer-274511/operations/DLDNXVBCWERBB2CY5WQEY3ZW'}
